<a href="https://colab.research.google.com/github/myresearchbvp/ERP-MCDA-Simulation/blob/main/Automated_ERP_PreSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# === 1. IMPORTS ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# === 2. DATASET (From Paper) ===
alts = ["ERP-A", "ERP-B", "ERP-C", "ERP-D"]
criteria = ["C1 (Functionality)", "C2 (Feasibility)", "C3 (Scalability)", "C4 (Usability)", "C5 (Integration)", "C6 (Support)"]

scores = {
    "ERP-A": [5.0, 3.0, 4.0, 4.0, 4.0, 5.0],
    "ERP-B": [4.0, 4.0, 3.0, 3.0, 4.0, 4.0],
    "ERP-C": [3.0, 5.0, 3.0, 4.0, 3.0, 3.0],
    "ERP-D": [4.0, 3.0, 5.0, 3.0, 4.0, 4.0]
}

# Corectat conform datelor din articol
weights = {
    "S1": [0.24, 0.16, 0.18, 0.14, 0.16, 0.12], # Baseline
    "S2": [0.20, 0.25, 0.15, 0.14, 0.14, 0.12], # Feasibility
    "S3": [0.20, 0.12, 0.28, 0.12, 0.16, 0.12]  # Growth
}

# === 3. CORE MCDA & VISUALIZATION FUNCTIONS ===
def run_mcda(w, current_scores, alt_list, crit_list):
    sc = {}
    ctb = {}
    for a in alt_list:
        s_raw = np.array(current_scores[a])
        w_arr = np.array(w)
        c_vals = s_raw * w_arr
        ctb[a] = c_vals
        sc[a] = np.sum(c_vals)

    rk = sorted(sc.items(), key=lambda x: x[1], reverse=True)
    win = rk[0][0]
    run = rk[1][0] if len(rk) > 1 else None
    margin = rk[0][1] - rk[1][1] if len(rk) > 1 else 0.0

    return sc, rk, win, run, margin, ctb

def plot_stacked_bar(ctb, crit_list, alt_list, title):
    fig, ax = plt.subplots(figsize=(10, 6))
    bottom = np.zeros(len(alt_list))

    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc949']

    for i, c in enumerate(crit_list):
        vals = [ctb[a][i] for a in alt_list]
        ax.bar(alt_list, vals, bottom=bottom, label=c, color=colors[i % len(colors)])
        bottom += vals

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Weighted Score')
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    for i, a in enumerate(alt_list):
        total = sum([ctb[a][j] for j in range(len(crit_list))])
        ax.text(i, total + 0.05, f'{total:.2f}', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

def generate_explanation(rk, margin, ctb, win, crit_list):
    if len(rk) < 2: return f"<h3>Recommendation: {win}</h3>"
    run = rk[1][0]
    win_ctb = ctb[win]
    best_crit = crit_list[np.argmax(win_ctb)]

    html = f"<div style='background-color: #f8f9fa; padding: 15px; border-left: 5px solid #28a745; margin-top: 15px;'>"
    html += f"<h3 style='margin-top:0;'>🤖 DSS Automated Explanation</h3>"
    html += f"<p>Based on the current criteria weights, the recommended system is <b>{win}</b> with a total score of <b>{rk[0][1]:.3f}</b>.</p>"
    html += f"<p>It outperforms the runner-up, <i>{run}</i> ({rk[1][1]:.3f}), by a margin of <b>{margin:.3f} points</b>. "
    html += f"The primary driver for {win}'s success is its strong performance in <b>{best_crit}</b>.</p></div>"
    return html

# === 4. EXTENDED DATA SETUP FOR SCALABILITY ===
extended_alts = alts.copy()
extended_scores = {k: v.copy() for k, v in scores.items()}

for i in range(4, 10):
    new_name = f"ERP-{chr(65+i)}"
    extended_alts.append(new_name)
    extended_scores[new_name] = [3.0] * len(criteria)

# === 5. INTERACTIVE DASHBOARD UI ===
slider_erps = widgets.IntSlider(value=4, min=2, max=10, step=1, description='No. of ERPs:', layout=widgets.Layout(width='300px'))
disclaimer_html = widgets.HTML(value="")

sliders_w = []
for i, c in enumerate(criteria):
    sl = widgets.FloatSlider(value=weights["S1"][i], min=0.0, max=0.5, step=0.01, description=c.split()[0]+":", layout=widgets.Layout(width='300px'))
    sliders_w.append(sl)

btn_s1 = widgets.Button(description="Load S1 (Balanced)", button_style='info')
btn_s2 = widgets.Button(description="Load S2 (Feasibility)", button_style='warning')
btn_s3 = widgets.Button(description="Load S3 (Growth)", button_style='success')

# Meniul Avansat
erp_selector = widgets.Dropdown(options=extended_alts[:4], description='Select ERP:', layout=widgets.Layout(width='250px'))
score_inputs = []
for c in criteria:
    inp = widgets.BoundedFloatText(value=3.0, min=1.0, max=5.0, step=0.1, description=c.split()[0]+":", layout=widgets.Layout(width='180px'))
    score_inputs.append(inp)

# Buton de salvare (elimină pâlpâirea)
btn_apply_scores = widgets.Button(description="💾 Apply & Run MCDA", button_style='primary', layout=widgets.Layout(width='200px', margin='10px 0px 0px 100px'))

def on_erp_select(change):
    selected_erp = erp_selector.value
    if selected_erp:
        for i, inp in enumerate(score_inputs):
            inp.value = extended_scores[selected_erp][i]

erp_selector.observe(on_erp_select, names='value')

def on_apply_click(b):
    selected_erp = erp_selector.value
    extended_scores[selected_erp] = [inp.value for inp in score_inputs]
    update_dashboard()

btn_apply_scores.on_click(on_apply_click)

out = widgets.Output()

def update_dashboard(*args):
    n = slider_erps.value

    current_options = extended_alts[:n]
    if erp_selector.options != tuple(current_options):
        erp_selector.options = current_options

    if n > 4:
        disclaimer_html.value = f"<div style='color: #0c5460; background-color: #d1ecf1; padding: 10px; border: 1px solid #bee5eb; border-radius: 5px; margin-bottom: 10px;'><b>Custom Simulation:</b> You are analyzing <b>{n} ERPs</b>. You can manually assign specific scores to the newly added systems by opening the <i>'Advanced: Manual Score Entry'</i> panel below.</div>"
    else:
        disclaimer_html.value = ""

    current_alts = extended_alts[:n]
    current_scores_dict = {a: extended_scores[a] for a in current_alts}

    w_curr = [sl.value for sl in sliders_w]
    if sum(w_curr) > 0:
        w_curr = [w / sum(w_curr) for w in w_curr]

    sc, rk, win, run, margin, ctb = run_mcda(w_curr, current_scores_dict, current_alts, criteria)

    with out:
        clear_output(wait=True)
        display(disclaimer_html)
        plot_stacked_bar(ctb, criteria, current_alts, f"Live MCDA Scores Breakdown ({n} Alternatives)")
        display(HTML(generate_explanation(rk, margin, ctb, win, criteria)))

        df_res = pd.DataFrame({"Final Score": [round(s, 3) for a,s in rk]}, index=[a for a,s in rk])
        display(df_res.T)

def load_preset(scenario_key):
    for i, val in enumerate(weights[scenario_key]):
        sliders_w[i].value = val

btn_s1.on_click(lambda x: load_preset("S1"))
btn_s2.on_click(lambda x: load_preset("S2"))
btn_s3.on_click(lambda x: load_preset("S3"))

slider_erps.observe(update_dashboard, names='value')
for sl in sliders_w:
    sl.observe(update_dashboard, names='value')

on_erp_select(None)

# === 6. DISPLAY LAYOUT ===
display(HTML("<h2>Automated & Explainable MCDA-based ERP Pre-Selection</h2>"))

ui_top = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>1. No. of Alternatives:</b>"), slider_erps]),
    widgets.VBox([widgets.HTML("<b>2. Predefined Scenarios:</b>"), widgets.HBox([btn_s1, btn_s2, btn_s3])])
])
ui_weights = widgets.VBox([widgets.HTML("<b>3. Adjust Criteria Weights:</b>")] + sliders_w)

ui_custom_scores = widgets.VBox([
    widgets.HTML("<i>Select an ERP, edit its scores, and click Apply to see changes:</i>"),
    erp_selector,
    widgets.HBox(score_inputs),
    btn_apply_scores
])
accordion = widgets.Accordion(children=[ui_custom_scores])
accordion.set_title(0, '⚙️ Advanced: Manual Score Entry (Edit Raw Data)')
accordion.selected_index = None

display(widgets.VBox([ui_top, ui_weights, accordion, out]))
update_dashboard()